# Utilização da extensão 'Parquet' - Transformando arquivos 'csv' em 'parquet': Criado pelo Nicholas Birochi

#### Importando bibliotecas essenciais para a Extração de Dados:

In [ ]:
# Importando bibliotecas:
from glob import glob  # Busca de arquivos por padrões...
import pandas as pd    # Análise e manipulação de dados...
import os              # Operações com arquivos e sistema...
import re              # Descarte de linhas com um padrão...

#### Criando as funções necessárias para esse processo:

In [ ]:
# Função para a conversão de cada uma das colunas em formato "string", assim não perdendo nenhum dado: 
def converter_colunas_mistas_para_str(df):
    for col in df.columns:
        if df[col].apply(type).nunique() > 1:
            df[col] = df[col].astype(str)
    return df

def ler_sem_colunas(path, colunas_excluir, **kwargs):
    colunas = pd.read_csv(path, nrows=0, **kwargs).columns.tolist()

    if all(isinstance(c, int) for c in colunas_excluir):
        colunas_ler = [col for i, col in enumerate(colunas) if i not in colunas_excluir]
    else:
        colunas_ler = [col for col in colunas if col not in colunas_excluir]

    return pd.read_csv(path, usecols=colunas_ler, **kwargs)

# Função final para a transformação de cada arquivo:
def converter_csv_para_parquet(pasta_csv, colunas_excluir=None, remover_caracteres=False):
    """
    Lê todos os .csv da pasta, ignora cabeçalhos/rodapés do relatório
    e gera parquet contendo os campos:
        partes[0] -> Cadastro (equivalente ao RG)
        partes[1] -> Nome
        partes[3] -> Data  (dd/mm/aaaa)
        partes[4] -> Hora  (hh:mm)
    renomeados para: Cadastro, Nome, Data, Hora.
    """

    REMOVER_PREFIXO_PRIMEIRA_COLUNA = remover_caracteres

    pasta_parquet = r'.\validation_base\parquet'
    os.makedirs(pasta_parquet, exist_ok=True)
    
    arquivos_csv = glob(os.path.join(pasta_csv, "*.csv"))
    
    for caminho in arquivos_csv:
        try:
            linhas_validas = []

            # Lê o arquivo linha a linha
            with open(caminho, "r", encoding="latin1") as f:
                for linha in f:
                    linha = linha.rstrip("\n")

                    # pula linhas vazias
                    if not linha.strip():
                        continue

                    partes = linha.split(",")

                    # precisa ter pelo menos 5 colunas para pegar 0,1,3,4
                    if len(partes) < 5:
                        continue

                    primeiro_campo = partes[0].strip()

                    # FILTRO: só considerar linhas de dados
                    # linhas de dados começam com algo numérico (ex: "00****")
                    # e não com textos de cabeçalho/rodapé
                    if not re.match(r"^\d", primeiro_campo):
                        continue

                    # ===== mapeamento correto dos campos =====
                    cadastro = partes[0].strip()  # Cadastro = RG
                    nome     = partes[1].strip()
                    data_str = partes[3].strip()
                    hora_str = partes[4].strip()

                    linhas_validas.append([cadastro, nome, data_str, hora_str])

            if not linhas_validas:
                print(f"⚠️ Nenhuma linha de dados encontrada em {caminho}")
                continue

            # monta DataFrame com as 4 colunas já com os nomes finais
            df = pd.DataFrame(linhas_validas, columns=["Cadastro", "Nome", "Data", "Hora"])

            # converte tudo pra string (mantendo o seu padrão)
            df = converter_colunas_mistas_para_str(df)

            # ===== AJUSTE OPCIONAL NA PRIMEIRA COLUNA (Cadastro) =====
            if REMOVER_PREFIXO_PRIMEIRA_COLUNA:
                df["Cadastro"] = df["Cadastro"].astype(str).str[2:]

            # gera nome do parquet
            nome_base = os.path.splitext(os.path.basename(caminho))[0]
            nome_arquivo = f"{nome_base}.parquet"
            caminho_salvar = os.path.join(pasta_parquet, nome_arquivo)

            df.to_parquet(caminho_salvar, index=False, engine='pyarrow')
            print(f"✅ Arquivo salvo: {caminho_salvar}")

        except Exception as e:
            print(f"⚠️ Erro ao processar {caminho}: {e}")


#### Inserindo os Endereços e nomes para dentro das funções:

In [ ]:
# Exemplo de uso para Horistas:
converter_csv_para_parquet(
    pasta_csv=R'.\validation_base\csv',
    colunas_excluir=None,
    remover_caracteres=True
)